# Wendy's Data 10/6/2025

Find sequences from Wendy's spreadsheet via isolate ID, then prepare data for tree-ing

In [1]:
# Housekeeping

import os
import pandas as pd
import numpy as np
import dateutil 
from datetime import datetime
from collections import defaultdict 
import importlib
import utils  
importlib.reload(utils)
from utils import * 

In [ ]:
# Make sure you have the correct paths

home = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu/"
downloads = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/"
# home = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu/"
# downloads = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/"
references = home + "references/"
originals = downloads + "Andersen/"
other = downloads + "Other/"
complete_files = other + "wendy_complete_09-22-2025/"
if not os.path.exists(complete_files): # checking if the directory exists or not
    os.makedirs(complete_files) # if the directory is not present then create it
# combined_files = downloads + "Combinations/NCBI_Virus_Andersen/" + date_range + "_Antarctica_North_America_South_America/"
# if not os.path.exists(combined_files): # checking if the directory exists or not
#     os.makedirs(combined_files) # if the directory is not present then create it
metadata_folder = originals + "avian-influenza/metadata/"

os.chdir(home + "references/")
state_ref = pd.read_csv("states_ref.csv")

os.chdir(other)
wendy_data = pd.read_excel("sample info for Alvin and Martha 20250922.xlsx", sheet_name="D1.1")

print(wendy_data)

isolates = wendy_data["accession"]
print(isolates)

# Get metadata from GitHub repo
os.chdir(metadata_folder)
metadata = pd.read_csv("SraRunTable_automated_normalized.tsv", delimiter="\t")
print(len(metadata)) 
print(metadata.columns)

# Find the name of the state sample was collected in
# metadata["name_state"] = metadata["geo_loc_name"].apply(lambda x: x.split("/")[1]) # if len(x.split("/")[1]) > 0 else x.split("/")[0])

# Convert the dates to date format so we can compare
# metadata["ReleaseDate"] = metadata["ReleaseDate"].apply(lambda x: dateutil.parser.parse(x).strftime("%Y-%m-%d"))
# metadata = metadata[metadata["ReleaseDate"] >= dateutil.parser.parse(start_date).strftime("%Y-%m-%d")] # Find only >= last date using Release Date from metadata 
# metadata = metadata[metadata["ReleaseDate"] <= dateutil.parser.parse(end_date).strftime("%Y-%m-%d")] # Find only <= update date using Release Date from metadata
metadata = metadata[metadata["is_retracted"] == False]
# display(metadata[metadata["geo_loc_name"] != "United States///"]) # ["geo_loc_name"])
# display(metadata)

         accession collection date Animal ID  sample ID        NVSL ID  year  \
0    25-003295-001      2024-01-05     25-14  24NE02028  25-003295-001  2024   
1    25-001721-001      2025-01-02      25-3  24HP01937  25-001721-001  2025   
2    25-007119-001      2025-01-05      25-7  24HP01933  25-007119-001  2025   
3    25-001719-006      2025-01-12   25-0050  24MM01926  25-001719-006  2025   
4    25-001719-003      2025-01-13     25-10  24NE01962  25-001719-003  2025   
..             ...             ...       ...        ...            ...   ...   
248  25-021407-027      2025-04-25     BBG10  25HP00825  25-021407-027  2025   
249  25-021407-030      2025-04-20       NaN  25WI00081  25-021407-030  2025   
250  25-021407-024      2025-07-13   25-1085  25WI00147  25-021407-024  2025   
251  25-021407-008      2025-02-13     25-91  24NE01845  25-021407-008  2025   
252  25-021407-023      2025-07-04   25-2263  25MM00595  25-021407-023  2025   

     month  class          group       

In [18]:
# Find the specific isolates we need
print("Wendy's data size:", len(wendy_data))

wendy_data["isolate"] = wendy_data["accession"]
df = wendy_data.merge(metadata, on="isolate")

print(df)
print("Found data size:", len(df)) 

# Don't bother with finding genotype, we know it's D1.1
genotype = "D1.1"

Wendy's data size: 253
         accession collection date Animal ID  sample ID        NVSL ID  year  \
0    25-003295-001      2024-01-05     25-14  24NE02028  25-003295-001  2024   
1    25-001721-001      2025-01-02      25-3  24HP01937  25-001721-001  2025   
2    25-007119-001      2025-01-05      25-7  24HP01933  25-007119-001  2025   
3    25-001719-006      2025-01-12   25-0050  24MM01926  25-001719-006  2025   
4    25-001719-003      2025-01-13     25-10  24NE01962  25-001719-003  2025   
..             ...             ...       ...        ...            ...   ...   
243  25-021407-027      2025-04-25     BBG10  25HP00825  25-021407-027  2025   
244  25-021407-030      2025-04-20       NaN  25WI00081  25-021407-030  2025   
245  25-021407-024      2025-07-13   25-1085  25WI00147  25-021407-024  2025   
246  25-021407-008      2025-02-13     25-91  24NE01845  25-021407-008  2025   
247  25-021407-023      2025-07-04   25-2263  25MM00595  25-021407-023  2025   

     month  clas

In [19]:


# # Double-check state with genbank_mapping
# os.chdir(metadata_folder)
# genbank_mapping = pd.read_csv("genbank_mapping.tsv", delimiter="\t")

# # Merge with genbank_mapping
# genbank_mapping = genbank_mapping.rename(columns={"sra_run":"Run"})
# metadata = metadata.merge(genbank_mapping, how="left")
# # Get the name of the state, unless it's not in genbank_mapping -- then get it from normalized metadata
# metadata["name_state_genbank"] = metadata["genbank_name"].apply(lambda x: x.split("/")[2] if x == x 
#                                                                 else x)
# metadata["name_state_genbank"] = metadata["name_state_genbank"].fillna(metadata["name_state"])

# forbidden_chars = [", ", ": "] # List of characters to replace
# Format: USA-[state abbreviation], e.g. USA-MD
df["Geo_Location"] = df["State"].apply( # lambda x: state_ref.loc[state_ref["Abbreviation"] == x.split("/")[2], 'Country'].iloc[0] + "-" + x.split("/")[2] if x.split("/")[2] in state_ref["Abbreviation"].values else state_ref.loc[state_ref["State"] == x.split("/")[2].replace("_", " "), 'Country'].iloc[0] + "-" + state_ref.loc[state_ref["State"] == x.split("/")[2].replace("_", " "), 'Abbreviation'].iloc[0] if x.split("/")[2].replace("_", " ") in state_ref["State"].values else x.split("/")[2].replace(": ", "-"))
    
                                                        lambda x: 
                                                        # If "x" has the state abbreviation (e.g. "MD")
                                                        state_ref.loc[state_ref["Abbreviation"].str.contains('|'.join(x.replace(": ", ",").replace(" ", "_").split(',')), regex=True), 'Country'].iloc[0] 
                                                        + "-" + 
                                                        x.split(" ")[-1]
                                                        if state_ref["Abbreviation"].str.contains("|".join((x.replace(": ", ",").replace(" ", "_").split(','))), regex=True).any()
                                                        # If "x" has the full state name (e.g. "Maryland")
                                                        else state_ref.loc[state_ref['State'].str.contains('|'.join(x.replace(": ", ",").replace(" ", "_").split(',')), regex=True), 'Country'].iloc[0]
                                                        + "-" + 
                                                        state_ref.loc[state_ref['State'].str.contains('|'.join(x.replace(": ", ",").replace(" ", "_").split(',')), regex=True), 'Abbreviation'].iloc[0] 
                                                        if state_ref["State"].str.contains("|".join((x.replace(": ", ",").replace(" ", "_").split(','))), regex=True).any()
                                                        # If "x" has neither the state abbreviation nor the full state name nor is "USA"
                                                        else 
                                                        "USA"
                                                        )

# If USA-, delete -
df["Geo_Location"] = df["Geo_Location"].apply(lambda x: x.split("-")[0] if x.split("-")[-1] == "" or x.split("-")[-1] == x.split("-")[0] else x)

# Rename variable back to metadata as we merge metadata and metadata_genbank
# metadata = metadata.merge(metadata_genbank, on="Run")

display(df)

,accession,collection date,Animal ID,sample ID,NVSL ID,year,month,class,group,scientific name,...,version,Sample Name,SRA Study,serotype,isolation_source,BioSample Accession,is_retracted,retraction_detection_date_utc,name_state,Geo_Location
0,25-003295-001,2024-01-05,25-14,24NE02028,25-003295-001,2024,Jan,avian,goose,Branta canadensis,...,1,25-003295-001,SRP557452,NaN,cloacal oropharyngeal swab pool,SRS24200661,False,NaN,,USA-ME
1,25-001721-001,2025-01-02,25-3,24HP01937,25-001721-001,2025,Jan,avian,goose,Branta canadensis,...,1,25-001721-001,SRP557452,NaN,cloacal oropharyngeal swab pool,SRS24003025,False,NaN,,USA-RI
2,25-007119-001,2025-01-05,25-7,24HP01933,25-007119-001,2025,Jan,avian,seabird,Larus marinus,...,1,25-007119-001,SRP557452,NaN,cloacal oropharyngeal swab pool,SRS24504220,False,NaN,,USA-RI
3,25-001719-006,2025-01-12,25-0050,24MM01926,25-001719-006,2025,Jan,avian,dabbling duck,Anas platyrhynchos,...,1,25-001719-006,SRP557452,NaN,cloacal oropharyngeal swab pool,SRS24003023,False,NaN,,USA-MA
4,25-001719-003,2025-01-13,25-10,24NE01962,25-001719-003,2025,Jan,avian,goose,Branta canadensis,...,1,25-001719-003,SRP557452,NaN,cloacal oropharyngeal swab pool,SRS24003021,False,NaN,,USA-MA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
243,25-021407-027,2025-04-25,BBG10,25HP00825,25-021407-027,2025,April,avian,seabird,Larus marinus,...,1,25-021407-027,SRP557452,NaN,swab cloacal,SRS26261388,False,NaN,,USA-MA
244,25-021407-030,2025-04-20,NaN,25WI00081,25-021407-030,2025,April,avian,seaduck,Somateria mollissima,...,1,25-021407-030,SRP557452,NaN,swab pool cloacal oropharyngeal,SRS26457501,False,NaN,,USA-MA
245,25-021407-024,2025-07-13,25-1085,25WI00147,25-021407-024,2025,July,avian,seabird,Larus marinus,...,1,25-021407-024,SRP557452,NaN,swab pool cloacal oropharyngeal,SRS26261385,False,NaN,,USA-MA
246,25-021407-008,2025-02-13,25-91,24NE01845,25-021407-008,2025,Feb,avian,owl,Bubo scandiacus,...,1,25-021407-008,SRP557452,NaN,swab pool cloacal oropharyngeal,SRS26261423,False,NaN,,USA-MA


In [20]:
# Get years from collection dates
df["years"] = df["collection date"].apply(lambda x: x.year) # Get year only from collection date

print(df["collection date"])

0     2024-01-05
1     2025-01-02
2     2025-01-05
3     2025-01-12
4     2025-01-13
         ...    
243   2025-04-25
244   2025-04-20
245   2025-07-13
246   2025-02-13
247   2025-07-04
Name: collection date, Length: 248, dtype: datetime64[ns]


In [21]:
# create a mask, where is True if the host does not exist
# mask = metadata["Host"].isna()

# choose between the original value and split isolate using the mask
# metadata["Host"] = np.where(mask, 
#                             metadata["isolate"].apply(lambda x: 
#                                                       x if x != x or "/" not in x or len(x.split("/")) < 2 # If NaN or split isolate doesn't exist or split isolate is too short
#                                                       else x.split("/")[1]), metadata["Host"]) # Provided that we have a long enough isolate with "/" in them, get the host

df["Host"] = df["common name"].apply(lambda x: x.lower() if x == x else x) # make sure all characters are lowercase

# Create animals ref if needed
unique_animals_all = sort_animals_andersen(df)

# Flatten unique_animals_all
every_unique_animal = []
for animal in unique_animals_all:
    every_unique_animal.append(animal)

# print(every_unique_animal)

unique_animals_set = list(set(every_unique_animal)) # Get rid of duplicates

os.chdir(references)
animals_ref = pd.read_csv("animals_ref.csv") # Upload animals ref

# If animal not in ref1, put in ref2
common_animals = []
# Check if animals in unique_animals_set are in ref1
for animal in unique_animals_set:
    for col in animals_ref.columns:
        if animal in animals_ref[col].values and animal == animal: # If animal exists in dataframe and isn't NaN
            common_animals.append(animal)

# If not in ref1, make a list of the new animals
different_animals = []
for animal in unique_animals_set:
    if animal not in common_animals:
        different_animals.append(animal)

print(different_animals)

# Add to dataframe
animals_df = animals_ref
# Make different_animals same length as dataframe, if shorter
if len(different_animals) < len(animals_df):
    number_of_times_to_add_nan = len(animals_df) - len(different_animals)
    for i in range(number_of_times_to_add_nan):
        different_animals.append(float('nan'))
# Unlikely for different_animals to be longer than the dataframe, but just in case
else:
    number_of_times_to_add_nan = len(different_animals) - len(animals_df)
    for i in range(number_of_times_to_add_nan):
        empty_rows = pd.DataFrame(np.nan, index=range(number_of_times_to_add_nan), columns=animals_df.columns)
        animals_df = pd.concat([animals_df, empty_rows], ignore_index=True)

animals_df["new"] = (different_animals)

print(animals_df)

animals_df.to_csv("animals_ref_to_sort.csv") # Make sure name is different to avoid overwriting the first reference 

print(df["Host"])
# print(metadata["isolate"])

# Get animals from animal reference
os.chdir(references)
animals_ref = pd.read_csv("animals_ref.csv")
fix_animals_andersen(df, animals_ref) # Get host type

print(df["Host_Type"])

[]
            wild_avian domestic_avian               cattle        feline  \
0     great_horned_owl       pheasant            dairy_cow           cat   
1         common_raven         turkey               cattle  domestic_cat   
2        cooper's_hawk        chicken  cattle milk product     feral_cat   
3         coopers_hawk          goose          bovine_milk        feline   
4              peafowl    guinea_fowl              bovine   domestic-cat   
..                 ...            ...                  ...           ...   
937     pintado_petrel            NaN                  NaN           NaN   
938  great blue heron             NaN                  NaN           NaN   
939   northern goshawk            NaN                  NaN           NaN   
940   peregrine falcon            NaN                  NaN           NaN   
941          fish crow            NaN                  NaN           NaN   

      other_mammal       human         other  new  
0       deer mouse  washington  

In [34]:
# Make names

# metadata["isolate_name"] = metadata["genbank_name"]

df = df.fillna("") # Make sure the entire name does not become "NaN"

df["isolate_name"] = np.where(df["sample name"] == "", "A/" + df["Host"].apply(lambda x: x.replace(" ", "_")) + "/" + df["State"]+ "/" + df["Sample Name"] + "/" + df["years"].apply(lambda x: str(x)), df["sample name"].apply(lambda x: x.split("(")[0]))


names = ">" + df["Run"] + "|" + df["isolate_name"] + "|H5N1|" + df["Geo_Location"] + "|" + df["collection date"].apply(lambda x: str(x).split(" ")[0]) + "|" + df["Host_Type"] + "|" + genotype

df["Name"] = names

df = df.drop_duplicates(subset="Run")

display(df["Name"])

0      >SRR32512599|A/Canada goose/ME/24NE02028/2024|...
1      >SRR32254418|A/Canada goose/RI/24HP01937/2025|...
2      >SRR32868883|A/Great black-backed gull/RI/24HP...
3      >SRR32254420|A/mallard duck/MA/24MM01926/2025|...
4      >SRR32254422|A/Canada goose/MA/24NE01962/2025|...
                             ...                        
243    >SRR35089320|A/Great black-backed gull/MA/25HP...
244    >SRR35319688|A/Common eider/MA/25WI00081/2025|...
245    >SRR35089323|A/Great black-backed gull/MA/25WI...
246    >SRR35089285|A/snowy owl/MA/24NE01845/2025|H5N...
247    >SRR35089324|A/great black-backed gull/MA/25MM...
Name: Name, Length: 247, dtype: object

In [35]:
# Get information to create the fasta files

fasta_folder = originals + "avian-influenza/fasta/"

os.chdir(fasta_folder)

segments = ["PB2", "PB1", "PA", "NS", "NP", "NA", "MP", "HA"]
pairs = []
fasta_files = {}

# Create pairs of genotypes and segments, e.g. B3.13_HA
for segment in segments:
    pair = genotype + "_" + segment
    pairs.append(pair)

for pair in pairs:
    fasta_files[pair] = [] # List to hold fasta files

for run in df["Run"].values: # For each run 
    for dirpath, dirs, files in os.walk(fasta_folder): # Find the fasta file
        for file in files:
            file_name = os.path.join(dirpath, file) # Get file name
            # print(file_name)
            if run in file_name: # Note that there will be ~8 files total with that run name
                # Make a fasta file and put it in the list
                with open(file_name) as f:
                    lines = f.readlines()
                    sequence = lines[1] 
                    # Each run/segment pair has one sequence -- it's placed into a file with other run/segment pairs with the same segment and genotype
                    header = df[df["Run"] == run].loc[:, "Name"].values[0]
                    genotype = genotype
                    # print(header)
                    # print(genotype)
                    # break 
                    segment = file_name.split("_")[-2]
                    # Find the pair that corresponds to 
                    pair_name = genotype + "_" + segment
                    this_specific_fasta = []
                    for pair in pairs:
                        # print(pair)
                        # print(pair_name)
                        if pair_name == pair:
                            this_specific_fasta.append(header)
                            this_specific_fasta.append(sequence)
                            fasta_files[pair].append(this_specific_fasta)
                f.close()
        break 

In [36]:
# Create fasta files 

os.chdir(complete_files)
names = []
for pair in fasta_files.keys():
    if len(fasta_files[pair]) > 0: # If this isn't empty
        output_path = complete_files + pair + "_wendy_09_22_2025.fasta"

        output_file = open(output_path, "w")
        for item in fasta_files[pair]:
            # for item in item:
            # item = fasta_files[pair]
            try:
                name = str(item[0].values[0]) # See if this is one we didn't have a collection date for
            except:
                name = str(item[0])
            print(name)
            name = name.replace(" ", "_")
            names.append(name)
            # First is header, second is sequence
            # print(value)
            output_file.write(name + "\n")
            output_file.write(item[1])
        output_file.close()

>SRR32512599|A/Canada goose/ME/24NE02028/2024|H5N1|USA-ME|2024-01-05|wild_avian|D1.1
>SRR32254418|A/Canada goose/RI/24HP01937/2025|H5N1|USA-RI|2025-01-02|wild_avian|D1.1
>SRR32868883|A/Great black-backed gull/RI/24HP01933/2025|H5N1|USA-RI|2025-01-05|wild_avian|D1.1
>SRR32254420|A/mallard duck/MA/24MM01926/2025|H5N1|USA-MA|2025-01-12|wild_avian|D1.1
>SRR32254422|A/Canada goose/MA/24NE01962/2025|H5N1|USA-MA|2025-01-13|wild_avian|D1.1
>SRR32254421|A/Canada goose/MA/24NE01963/2025|H5N1|USA-MA|2025-01-13|wild_avian|D1.1
>SRR32973996|A/Canada goose/MA/24NE01964/2025|H5N1|USA-MA|2025-01-13|wild_avian|D1.1
>SRR32254419|A/Canada goose/MA/24MM01937/2025|H5N1|USA-MA|2025-01-14|wild_avian|D1.1
>SRR32804638|A/Canada goose/MA/24HP02020/2025|H5N1|USA-MA|2025-01-14|wild_avian|D1.1
>SRR32512623|A/great blue heron /MA/24HP02025/2025|H5N1|USA-MA|2025-01-14|wild_avian|D1.1
>SRR32512543|A/Canada goose/MA/24MM01942/2025|H5N1|USA-MA|2025-01-16|wild_avian|D1.1
>SRR32512548|A/Canada goose/MA/24NE01958/2025|H5N